# Interpretability in NAMpy

This notebook demonstrates the interpretability features of NAMpy models, including:

1. **Shape Functions**: Understanding individual feature contributions
2. **Feature Importance**: Quantifying feature relevance
3. **Interaction Effects**: Exploring feature interactions (where applicable)
4. **Visualization Techniques**: Best practices for communicating model insights

One of the key advantages of Neural Additive Models (NAMs) over black-box models is their inherent interpretability. The additive structure means:

$$f(x) = \beta_0 + \sum_{j=1}^{p} f_j(x_j)$$

Each feature $x_j$ has its own shape function $f_j$ that can be visualized independently.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['font.size'] = 11

## 1. Understanding Shape Functions

Shape functions are the core interpretability tool in NAMs. They show how each feature value maps to a contribution to the prediction.

In [ ]:
# Load California Housing dataset
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

print("Dataset shape:", X.shape)
print("\nFeature names:")
for i, name in enumerate(data.feature_names):
    print(f"  {i}: {name}")

In [ ]:
# Use a subset for faster training
X_subset = X.sample(n=5000, random_state=42)
y_subset = y[X_subset.index]

X_train, X_test, y_train, y_test = train_test_split(
    X_subset.values, y_subset, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
from nampy.models import NAMRegressor

# Train a NAM model
model = NAMRegressor(
    numerical_preprocessing="ple",
    n_bins=50,
    layer_sizes=[64, 32],
    dropout=0.2
)

model.fit(
    X_train, y_train,
    max_epochs=100,
    lr=1e-3,
    patience=10
)

# Evaluate
from sklearn.metrics import r2_score, mean_squared_error
y_pred = model.predict(X_test)
print(f"\nModel Performance:")
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

### 1.1 Extracting and Visualizing Shape Functions

Shape functions show the partial dependence of predictions on each feature. Let's extract and visualize them.

In [ ]:
def plot_shape_functions(model, X, feature_names, ncols=4):
    """
    Plot shape functions for all features.
    
    Parameters:
    -----------
    model : NAMpy model
        Trained NAMpy model
    X : array-like
        Input data for generating predictions
    feature_names : list
        Names of features
    ncols : int
        Number of columns in the plot grid
    """
    n_features = len(feature_names)
    nrows = int(np.ceil(n_features / ncols))
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3*nrows))
    axes = axes.flatten()
    
    # Get feature contributions from the model
    # NAMpy models store shape functions internally
    for i in range(n_features):
        ax = axes[i]
        
        # Create a range of values for this feature
        feature_vals = np.linspace(X[:, i].min(), X[:, i].max(), 100)
        
        # Create input with this feature varying and others at median
        X_partial = np.tile(np.median(X, axis=0), (100, 1))
        X_partial[:, i] = feature_vals
        
        # Get predictions
        preds = model.predict(X_partial)
        
        # Center the predictions
        preds_centered = preds - preds.mean()
        
        # Plot
        ax.plot(feature_vals, preds_centered, 'b-', linewidth=2)
        
        # Add histogram of feature values
        ax2 = ax.twinx()
        ax2.hist(X[:, i], bins=30, alpha=0.2, color='gray')
        ax2.set_ylabel('')
        ax2.set_yticks([])
        
        ax.set_xlabel(feature_names[i])
        ax.set_ylabel('Contribution')
        ax.set_title(f'Shape Function: {feature_names[i]}')
        ax.axhline(y=0, color='r', linestyle='--', alpha=0.5)
    
    # Hide empty subplots
    for i in range(n_features, len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout()
    return fig

# Plot shape functions
fig = plot_shape_functions(model, X_train, data.feature_names)
plt.suptitle('Shape Functions for California Housing Prediction', y=1.02, fontsize=14)
plt.show()

### 1.2 Interpreting Shape Functions

Let's look at some key features in more detail:

In [ ]:
def detailed_shape_function(model, X, feature_idx, feature_name, ax=None):
    """
    Create a detailed visualization of a single shape function.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))
    
    # Create range of values
    feature_vals = np.linspace(X[:, feature_idx].min(), X[:, feature_idx].max(), 200)
    
    # Create partial dependence data
    X_partial = np.tile(np.median(X, axis=0), (200, 1))
    X_partial[:, feature_idx] = feature_vals
    
    # Get predictions
    preds = model.predict(X_partial)
    preds_centered = preds - preds.mean()
    
    # Main shape function plot
    ax.plot(feature_vals, preds_centered, 'b-', linewidth=2.5, label='Shape Function')
    ax.fill_between(feature_vals, preds_centered, alpha=0.2)
    
    # Add rug plot for data density
    ax.scatter(X[:, feature_idx], np.full(len(X), ax.get_ylim()[0] + 0.01), 
               alpha=0.1, s=1, c='black')
    
    # Mark key statistics
    median_val = np.median(X[:, feature_idx])
    ax.axvline(x=median_val, color='green', linestyle='--', alpha=0.7, label=f'Median: {median_val:.2f}')
    
    ax.axhline(y=0, color='red', linestyle='-', alpha=0.3)
    ax.set_xlabel(feature_name, fontsize=12)
    ax.set_ylabel('Contribution to Prediction', fontsize=12)
    ax.set_title(f'Detailed Shape Function: {feature_name}', fontsize=14)
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    return ax

# Analyze MedInc (Median Income) - usually the most important feature
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MedInc (index 0)
detailed_shape_function(model, X_train, 0, 'MedInc (Median Income)', axes[0])

# AveRooms (index 2)
detailed_shape_function(model, X_train, 2, 'AveRooms (Avg Rooms per Household)', axes[1])

plt.tight_layout()
plt.show()

## 2. Feature Importance

Feature importance quantifies how much each feature contributes to predictions. We can compute this in several ways.

In [ ]:
def compute_feature_importance(model, X, method='variance'):
    """
    Compute feature importance based on shape function variance.
    
    Parameters:
    -----------
    model : NAMpy model
        Trained model
    X : array-like
        Input data
    method : str
        'variance' - variance of partial predictions
        'range' - range of partial predictions
        
    Returns:
    --------
    importances : array
        Feature importance scores
    """
    n_features = X.shape[1]
    importances = np.zeros(n_features)
    
    for i in range(n_features):
        # Create partial dependence data
        feature_vals = np.linspace(X[:, i].min(), X[:, i].max(), 100)
        X_partial = np.tile(np.median(X, axis=0), (100, 1))
        X_partial[:, i] = feature_vals
        
        # Get predictions
        preds = model.predict(X_partial)
        
        if method == 'variance':
            importances[i] = np.var(preds)
        elif method == 'range':
            importances[i] = preds.max() - preds.min()
    
    # Normalize
    importances = importances / importances.sum()
    return importances

# Compute importance
importance = compute_feature_importance(model, X_train, method='variance')

# Create a bar plot
fig, ax = plt.subplots(figsize=(10, 6))

# Sort by importance
sorted_idx = np.argsort(importance)[::-1]
sorted_importance = importance[sorted_idx]
sorted_names = [data.feature_names[i] for i in sorted_idx]

colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(sorted_names)))
bars = ax.barh(range(len(sorted_names)), sorted_importance, color=colors)

ax.set_yticks(range(len(sorted_names)))
ax.set_yticklabels(sorted_names)
ax.invert_yaxis()
ax.set_xlabel('Relative Importance', fontsize=12)
ax.set_title('Feature Importance (based on shape function variance)', fontsize=14)

# Add percentage labels
for i, (bar, imp) in enumerate(zip(bars, sorted_importance)):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{imp*100:.1f}%', va='center', fontsize=10)

plt.tight_layout()
plt.show()

### 2.1 Comparing Importance Methods

In [ ]:
# Compare different importance methods
importance_variance = compute_feature_importance(model, X_train, method='variance')
importance_range = compute_feature_importance(model, X_train, method='range')

# Create comparison DataFrame
importance_df = pd.DataFrame({
    'Feature': data.feature_names,
    'Variance-based': importance_variance,
    'Range-based': importance_range
}).sort_values('Variance-based', ascending=False)

print("Feature Importance Comparison:")
print("="*60)
print(importance_df.to_string(index=False))

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(data.feature_names))
width = 0.35

# Sort by variance-based importance
sorted_idx = np.argsort(importance_variance)[::-1]

bars1 = ax.bar(x - width/2, importance_variance[sorted_idx], width, 
               label='Variance-based', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, importance_range[sorted_idx], width, 
               label='Range-based', color='darkorange', alpha=0.8)

ax.set_xlabel('Feature')
ax.set_ylabel('Relative Importance')
ax.set_title('Comparison of Feature Importance Methods')
ax.set_xticks(x)
ax.set_xticklabels([data.feature_names[i] for i in sorted_idx], rotation=45, ha='right')
ax.legend()

plt.tight_layout()
plt.show()

## 3. Local Interpretations

While shape functions provide global interpretability, we can also explain individual predictions.

In [ ]:
def explain_prediction(model, x, feature_names, X_train):
    """
    Explain a single prediction by computing feature contributions.
    
    Parameters:
    -----------
    model : NAMpy model
        Trained model
    x : array-like
        Single sample to explain
    feature_names : list
        Names of features
    X_train : array-like
        Training data for baseline computation
        
    Returns:
    --------
    contributions : dict
        Feature contributions
    """
    # Baseline prediction (at median values)
    baseline = np.median(X_train, axis=0)
    baseline_pred = model.predict(baseline.reshape(1, -1))[0]
    
    # Actual prediction
    actual_pred = model.predict(x.reshape(1, -1))[0]
    
    # Compute contributions by changing one feature at a time
    contributions = {}
    for i, name in enumerate(feature_names):
        # Prediction with just this feature changed from baseline
        x_partial = baseline.copy()
        x_partial[i] = x[i]
        partial_pred = model.predict(x_partial.reshape(1, -1))[0]
        
        contributions[name] = partial_pred - baseline_pred
    
    return {
        'baseline_prediction': baseline_pred,
        'actual_prediction': actual_pred,
        'contributions': contributions
    }

# Explain a specific prediction
sample_idx = 0
x_sample = X_test[sample_idx]
explanation = explain_prediction(model, x_sample, data.feature_names, X_train)

print(f"Explaining prediction for sample {sample_idx}:")
print(f"\nActual house value: ${y_test[sample_idx]*100000:.0f}")
print(f"Predicted house value: ${explanation['actual_prediction']*100000:.0f}")
print(f"Baseline prediction: ${explanation['baseline_prediction']*100000:.0f}")
print(f"\nFeature Contributions:")
print("-" * 40)
for name, contrib in sorted(explanation['contributions'].items(), key=lambda x: abs(x[1]), reverse=True):
    sign = '+' if contrib >= 0 else ''
    print(f"{name:15s}: {sign}${contrib*100000:>10,.0f}")

In [ ]:
def waterfall_plot(explanation, feature_names, ax=None):
    """
    Create a waterfall plot for feature contributions.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 6))
    
    contributions = explanation['contributions']
    baseline = explanation['baseline_prediction']
    
    # Sort by absolute contribution
    sorted_contribs = sorted(contributions.items(), key=lambda x: abs(x[1]), reverse=True)
    names = [x[0] for x in sorted_contribs]
    values = [x[1] for x in sorted_contribs]
    
    # Calculate cumulative sum for waterfall
    cumsum = baseline + np.cumsum([0] + values[:-1])
    
    colors = ['green' if v >= 0 else 'red' for v in values]
    
    # Plot bars
    ax.barh(range(len(names)), values, left=cumsum, color=colors, alpha=0.7)
    
    # Add baseline and final value markers
    ax.axvline(x=baseline, color='black', linestyle='--', label=f'Baseline: {baseline:.3f}')
    ax.axvline(x=explanation['actual_prediction'], color='blue', linestyle='-', 
               label=f'Prediction: {explanation["actual_prediction"]:.3f}')
    
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names)
    ax.invert_yaxis()
    ax.set_xlabel('Prediction Value')
    ax.set_title('Waterfall Plot: Feature Contributions')
    ax.legend(loc='lower right')
    
    return ax

# Create waterfall plot
fig, ax = plt.subplots(figsize=(12, 6))
waterfall_plot(explanation, data.feature_names, ax)
plt.tight_layout()
plt.show()

## 4. Classification Interpretability

Let's also look at interpretability for classification models.

In [ ]:
from nampy.models import NAMClassifier
from sklearn.metrics import accuracy_score

# Load breast cancer dataset
cancer_data = load_breast_cancer()
X_cancer = cancer_data.data
y_cancer = cancer_data.target
cancer_features = cancer_data.feature_names

# Split data
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cancer, y_cancer, test_size=0.2, random_state=42
)

# Train classifier
clf = NAMClassifier(
    numerical_preprocessing="standardization",
    layer_sizes=[32, 16],
    dropout=0.3
)

clf.fit(
    X_train_c, y_train_c,
    max_epochs=100,
    lr=1e-3,
    patience=10
)

y_pred_c = clf.predict(X_test_c)
print(f"Classification Accuracy: {accuracy_score(y_test_c, y_pred_c):.4f}")

In [ ]:
# Compute feature importance for classifier
clf_importance = compute_feature_importance(clf, X_train_c, method='variance')

# Get top 10 features
top_k = 10
sorted_idx = np.argsort(clf_importance)[::-1][:top_k]

fig, ax = plt.subplots(figsize=(10, 6))

colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, top_k))
bars = ax.barh(range(top_k), clf_importance[sorted_idx], color=colors)

ax.set_yticks(range(top_k))
ax.set_yticklabels([cancer_features[i] for i in sorted_idx])
ax.invert_yaxis()
ax.set_xlabel('Relative Importance')
ax.set_title('Top 10 Most Important Features for Breast Cancer Classification')

plt.tight_layout()
plt.show()

In [ ]:
# Plot shape functions for top features in classification
top_features = sorted_idx[:4]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, feat_idx in enumerate(top_features):
    ax = axes[i]
    
    # Create range of values
    feature_vals = np.linspace(X_train_c[:, feat_idx].min(), 
                                X_train_c[:, feat_idx].max(), 100)
    
    # Create partial dependence data
    X_partial = np.tile(np.median(X_train_c, axis=0), (100, 1))
    X_partial[:, feat_idx] = feature_vals
    
    # Get probability predictions
    probs = clf.predict_proba(X_partial)[:, 1]  # Probability of benign (class 1)
    
    ax.plot(feature_vals, probs, 'b-', linewidth=2)
    ax.fill_between(feature_vals, 0.5, probs, where=probs > 0.5, 
                    alpha=0.3, color='green', label='Benign tendency')
    ax.fill_between(feature_vals, probs, 0.5, where=probs < 0.5, 
                    alpha=0.3, color='red', label='Malignant tendency')
    
    ax.axhline(y=0.5, color='black', linestyle='--', alpha=0.5)
    ax.set_xlabel(cancer_features[feat_idx])
    ax.set_ylabel('P(Benign)')
    ax.set_title(f'Shape Function: {cancer_features[feat_idx]}')
    ax.set_ylim(0, 1)
    ax.legend(loc='best')

plt.suptitle('Shape Functions for Top Features in Classification', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Model Comparison: Interpretability across Different Architectures

Let's compare how different NAMpy models learn shape functions for the same data.

In [ ]:
from nampy.models import NAMRegressor, NBMRegressor

# Train different models
models = {}

# NAM
models['NAM'] = NAMRegressor(numerical_preprocessing="ple", n_bins=50)
models['NAM'].fit(X_train, y_train, max_epochs=100, lr=1e-3, patience=10)

# NBM
models['NBM'] = NBMRegressor(numerical_preprocessing="ple", n_bins=50)
models['NBM'].fit(X_train, y_train, max_epochs=100, lr=1e-3, patience=10)

print("Models trained successfully!")

In [ ]:
def compare_shape_functions(models, X, feature_idx, feature_name):
    """
    Compare shape functions across different models.
    """
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Create range of values
    feature_vals = np.linspace(X[:, feature_idx].min(), X[:, feature_idx].max(), 100)
    
    colors = ['blue', 'orange', 'green', 'red', 'purple']
    
    for i, (name, model) in enumerate(models.items()):
        # Create partial dependence data
        X_partial = np.tile(np.median(X, axis=0), (100, 1))
        X_partial[:, feature_idx] = feature_vals
        
        # Get predictions
        preds = model.predict(X_partial)
        preds_centered = preds - preds.mean()
        
        ax.plot(feature_vals, preds_centered, color=colors[i], linewidth=2, label=name)
    
    ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
    ax.set_xlabel(feature_name, fontsize=12)
    ax.set_ylabel('Contribution', fontsize=12)
    ax.set_title(f'Shape Function Comparison: {feature_name}', fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    return fig

# Compare shape functions for MedInc
fig = compare_shape_functions(models, X_train, 0, 'MedInc (Median Income)')
plt.tight_layout()
plt.show()

In [ ]:
# Compare multiple features
feature_indices = [0, 2, 5, 7]  # MedInc, AveRooms, Population, Longitude

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

colors = ['blue', 'orange']

for idx, feat_idx in enumerate(feature_indices):
    ax = axes[idx]
    
    feature_vals = np.linspace(X_train[:, feat_idx].min(), 
                                X_train[:, feat_idx].max(), 100)
    
    for i, (name, model) in enumerate(models.items()):
        X_partial = np.tile(np.median(X_train, axis=0), (100, 1))
        X_partial[:, feat_idx] = feature_vals
        
        preds = model.predict(X_partial)
        preds_centered = preds - preds.mean()
        
        ax.plot(feature_vals, preds_centered, color=colors[i], linewidth=2, label=name)
    
    ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
    ax.set_xlabel(data.feature_names[feat_idx])
    ax.set_ylabel('Contribution')
    ax.set_title(f'{data.feature_names[feat_idx]}')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)

plt.suptitle('Shape Function Comparison Across Models', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6. Best Practices for Interpretability

### Summary of Interpretability Techniques

| Technique | Use Case | Pros | Cons |
|-----------|----------|------|------|
| Shape Functions | Understanding feature effects | Global view, intuitive | Assumes additive structure |
| Feature Importance | Feature selection, model summary | Quick overview | May miss interactions |
| Local Explanations | Explaining individual predictions | Actionable insights | Time consuming for many samples |
| Model Comparison | Validating learned relationships | Robust insights | Computationally expensive |

### Tips for Effective Interpretation

1. **Always validate shape functions** - Compare with domain knowledge
2. **Check data density** - Shape functions are less reliable in sparse regions
3. **Use multiple models** - Consistent patterns across models are more trustworthy
4. **Consider interactions** - Pure additive models may miss important interactions
5. **Communicate uncertainty** - Show confidence intervals when possible

In [ ]:
# Summary statistics
print("=" * 60)
print("INTERPRETABILITY ANALYSIS SUMMARY")
print("=" * 60)

print("\n1. MODEL PERFORMANCE:")
for name, model in models.items():
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    print(f"   {name}: R² = {r2:.4f}, RMSE = {rmse:.4f}")

print("\n2. TOP 5 IMPORTANT FEATURES:")
for i, idx in enumerate(np.argsort(importance_variance)[::-1][:5]):
    print(f"   {i+1}. {data.feature_names[idx]}: {importance_variance[idx]*100:.1f}%")

print("\n3. KEY INSIGHTS:")
print("   - MedInc (Median Income) is the dominant predictor")
print("   - Geographic features (Lat, Long) capture location effects")
print("   - Shape functions reveal non-linear relationships")
print("   - Models agree on general patterns but differ in details")

## Conclusion

In this notebook, we explored the interpretability features of NAMpy:

1. **Shape Functions** provide visual understanding of how each feature affects predictions
2. **Feature Importance** quantifies the relative contribution of each feature
3. **Local Explanations** help understand individual predictions
4. **Model Comparison** validates that learned patterns are robust

The key advantage of NAMs is their inherent interpretability - unlike post-hoc explanation methods for black-box models, NAM interpretations directly reflect how the model makes predictions.

### Next Steps

- Explore the other notebooks for more examples:
  - `01_basic_regression.ipynb` - Regression fundamentals
  - `02_classification.ipynb` - Classification tasks
  - `03_distributional_regression.ipynb` - Uncertainty modeling
  - `04_model_comparison.ipynb` - Comparing different architectures